<div align="right"><small>
<b>Pablo Emilio Gómez Gómez</b><br>
Proyecto Integrador III – Analítica de Datos<br>
Docente: Andrés Felipe Callejas Jaramillo<br>
Código: PREICA2602B020038<br>
Institución Universitaria Digital de Antioquia<br>
Bogotá D. C., 2 de noviembre de 2026
</small></div>

# AE3 – Transformación y Enriquecimiento de Datos

Este notebook da continuidad a EA1 y EA2 con el mismo dataset `mstz/heart_failure`, configuración `death`, split `train`. Conserva una copia sin modificar, documenta controles de calidad, transforma tipos, incorpora variables derivadas, responde descriptivamente la necesidad y exporta automáticamente los archivos requeridos.

> **Alcance:** los resultados son descriptivos dentro de este conjunto. No se realizan diagnósticos, predicciones, inferencias causales ni recomendaciones médicas.

> **Criterio pedagógico:** el código se explica por líneas o bloques breves. Los comentarios numerados indican cómo entender la instrucción y para qué se utiliza.


## 1. Necesidad, pregunta de EA2 y operacionalización para AE3

**Necesidad de información:** organizar y describir las variables demográficas, clínicas y de estilo de vida disponibles para comparar los registros según si se documentó o no muerte durante el seguimiento.

**Pregunta central de EA2:** ¿Cómo se caracterizan y qué diferencias descriptivas se observan en las variables demográficas, clínicas y de estilo de vida disponibles entre los pacientes que registraron y los que no registraron muerte durante el seguimiento del dataset `mstz/heart_failure`?

Para convertir esa pregunta amplia en tablas verificables, AE3 la operacionaliza en cinco preguntas descriptivas, sin modificar su objetivo:

1. ¿Cómo se distribuyen los 299 registros entre muerte registrada y no registrada durante el seguimiento?
2. ¿Qué diferencias descriptivas se observan en las variables cuantitativas entre ambos grupos?
3. ¿Qué diferencias descriptivas se observan en las condiciones e indicadores binarios entre ambos grupos?
4. ¿Cómo se distribuye la proporción de registros con muerte registrada según grupos descriptivos de edad?
5. ¿Cómo se distribuye la proporción de registros con muerte registrada según el número de condiciones registradas (anemia, diabetes y presión arterial alta)?

**Limitación explícita:** `days_in_study` es tiempo de seguimiento y se describe como tal; no se interpreta como causa ni predictor del desenlace.


In [1]:
# 1. Importamos Path desde pathlib.
# 2. Sirve para construir rutas relativas que funcionen en Windows y en GitHub Actions.
from pathlib import Path

# 3. Importamos sys.
# 4. Sirve para agregar la carpeta AE3 al buscador de módulos de Python.
import sys

# 5. Importamos pandas con el alias pd.
# 6. Sirve para manejar y visualizar DataFrames.
import pandas as pd

# 7. Importamos display desde IPython.
# 8. Sirve para mostrar tablas de forma legible dentro de Jupyter.
from IPython.display import display

# 9. Tomamos la carpeta actual y buscamos hacia arriba la raíz que contiene AE3/src.
# 10. Sirve para que el notebook funcione tanto localmente como en GitHub Actions.
actual = Path.cwd().resolve()
RAIZ = next((p for p in [actual, *actual.parents] if (p / "AE3" / "src").exists()), None)
assert RAIZ is not None, "No se encontró la raíz del proyecto."

# 11. Agregamos la carpeta AE3 a sys.path.
# 12. Sirve para importar el paquete src sin usar rutas personales del computador.
sys.path.insert(0, str(RAIZ / "AE3"))

# 13. Importamos las funciones construidas para esta entrega.
# 14. Sirve para mantener el notebook legible y separar responsabilidades en archivos .py.
from src.carga_datos import cargar_heart_failure
from src.transformacion import transformar_y_validar
from src.enriquecimiento import enriquecer
from src.analisis import construir_tablas
from src.reportes import exportar_evidencias, construir_diccionario

print("Raíz del proyecto:", RAIZ)


Raíz del proyecto: C:\Users\Asus\AppData\Local\PI3_AE3_TRABAJO\EA2_HeartFailure_Pablo_Gomez


## 2. Carga del dataset y conservación de una copia sin modificar

In [2]:
# 1. Cargamos el mismo dataset usado desde EA1 y EA2.
# 2. Sirve para mantener continuidad con mstz/heart_failure, configuración death y split train.
df_entrada, metadatos = cargar_heart_failure(preferir_huggingface=True)

# 3. Creamos una copia profunda antes de transformar.
# 4. Sirve para conservar evidencia de cómo llegaron los datos y comparar antes/después.
df_original = df_entrada.copy(deep=True)

# 5. Mostramos procedencia, dimensiones y cinco registros.
# 6. Sirve para dejar evidencia visible de la carga efectuada en esta ejecución.
print(metadatos)
print("Dimensiones originales:", df_original.shape)
display(df_original.head())


C:\Users\Asus\AppData\Local\PI3_AE3_TRABAJO\.venv_ae3\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'dataset': 'mstz/heart_failure', 'configuracion': 'death', 'split': 'train', 'fuente_usada': 'Hugging Face', 'error_huggingface': None}
Dimensiones originales: (299, 13)


,age,has_anaemia,creatinine_phosphokinase_concentration_in_blood,has_diabetes,heart_ejection_fraction,has_high_blood_pressure,platelets_concentration_in_blood,serum_creatinine_concentration_in_blood,serum_sodium_concentration_in_blood,is_male,is_smoker,days_in_study,is_dead
0,75,False,582.0,False,20.0,True,265000.00,1.9,130.0,True,False,4,1
1,55,False,7861.0,False,38.0,False,263358.03,1.1,136.0,True,False,6,1
2,65,False,146.0,False,20.0,False,162000.00,1.3,129.0,True,True,7,1
3,50,True,111.0,False,20.0,False,210000.00,1.9,137.0,True,False,7,1
4,65,True,160.0,True,20.0,False,327000.00,2.7,116.0,False,False,8,1


## 3. Relación entre preguntas, variables y resultado esperado

In [3]:
# 1. Construimos una tabla que conecta cada pregunta con sus variables y salida.
# 2. Sirve para demostrar que cada transformación y cada resultado aporta a la necesidad del proyecto.
preguntas = pd.DataFrame([
    {"pregunta": "P1. Distribución del desenlace registrado", "variables": "is_dead / estado_seguimiento", "resultado_esperado": "Conteos y porcentajes de ambos grupos"},
    {"pregunta": "P2. Diferencias en variables cuantitativas", "variables": "age, CPK, fracción de eyección, plaquetas, creatinina, sodio y days_in_study", "resultado_esperado": "n, media, mediana, desviación, CV, cuartiles y rango observado por grupo"},
    {"pregunta": "P3. Diferencias en indicadores binarios", "variables": "anemia, diabetes, presión arterial alta, sexo y tabaquismo", "resultado_esperado": "Conteos y porcentajes dentro de cada grupo"},
    {"pregunta": "P4. Grupos descriptivos de edad", "variables": "age, grupo_edad, is_dead", "resultado_esperado": "Total y porcentaje con muerte registrada por grupo de edad"},
    {"pregunta": "P5. Número de condiciones registradas", "variables": "anemia, diabetes, presión arterial alta, condiciones_registradas, is_dead", "resultado_esperado": "Total y porcentaje con muerte registrada por nivel de condiciones"},
])
display(preguntas)


,pregunta,variables,resultado_esperado
0,P1. Distribución del desenlace registrado,is_dead / estado_seguimiento,Conteos y porcentajes de ambos grupos
1,P2. Diferencias en variables cuantitativas,"age, CPK, fracción de eyección, plaquetas, cre...","n, media, mediana, desviación, CV, cuartiles y..."
2,P3. Diferencias en indicadores binarios,"anemia, diabetes, presión arterial alta, sexo ...",Conteos y porcentajes dentro de cada grupo
3,P4. Grupos descriptivos de edad,"age, grupo_edad, is_dead",Total y porcentaje con muerte registrada por g...
4,P5. Número de condiciones registradas,"anemia, diabetes, presión arterial alta, condi...",Total y porcentaje con muerte registrada por n...


## 4. Transformación y validación de calidad

In [4]:
# 1. Ejecutamos la transformación y los controles sobre una copia del original.
# 2. Sirve para normalizar tipos sin borrar registros y documentar cada control solicitado.
df_transformado, control_calidad, comparacion, comparacion_tipos, extremos_iqr = transformar_y_validar(df_original)

# 3. Mostramos los controles y la comparación estructural.
# 4. Sirve para informar incluso los controles que encuentran cero problemas.
display(control_calidad)
display(comparacion)
display(comparacion_tipos)

# 5. Mostramos los posibles extremos por regla IQR.
# 6. Sirve para identificarlos sin eliminarlos automáticamente: extremo estadístico no equivale a error.
display(extremos_iqr)


,control,resultado,afectados,decision
0,Valores nulos antes,0,0,No imputar si el resultado es 0.
1,Filas duplicadas exactas antes,0,0,No eliminar si el resultado es 0.
2,Valores fuera de 0/1 en indicadores binarios,0,0,No se modifica ningún registro si el resultado...
3,Rango estructural: edad_no_positiva,0,0,Revisar únicamente si el resultado es mayor qu...
4,Rango estructural: fraccion_eyeccion_fuera_0_100,0,0,Revisar únicamente si el resultado es mayor qu...
5,Rango estructural: plaquetas_negativas,0,0,Revisar únicamente si el resultado es mayor qu...
6,Rango estructural: creatinina_serica_negativa,0,0,Revisar únicamente si el resultado es mayor qu...
7,Rango estructural: sodio_serico_negativo,0,0,Revisar únicamente si el resultado es mayor qu...
8,Rango estructural: cpk_negativa,0,0,Revisar únicamente si el resultado es mayor qu...
9,Rango estructural: seguimiento_negativo,0,0,Revisar únicamente si el resultado es mayor qu...


,indicador,antes,despues_transformacion
0,filas,299,299
1,columnas,13,13
2,nulos_totales,0,0
3,duplicados_exactos,0,0


,variable,tipo_antes,tipo_despues_transformacion
0,age,int64,int64
1,has_anaemia,bool,int8
2,creatinine_phosphokinase_concentration_in_blood,float64,float64
3,has_diabetes,bool,int8
4,heart_ejection_fraction,float64,float64
5,has_high_blood_pressure,bool,int8
6,platelets_concentration_in_blood,float64,float64
7,serum_creatinine_concentration_in_blood,float64,float64
8,serum_sodium_concentration_in_blood,float64,float64
9,is_male,bool,int8


,variable,q1,q3,iqr,limite_inferior_iqr,limite_superior_iqr,valores_extremos_iqr,tratamiento
0,age,51.0,70.0,19.0,22.50,98.50,0,"Se conservan; son extremos estadísticos, no er..."
1,creatinine_phosphokinase_concentration_in_blood,116.5,582.0,465.5,-581.75,1280.25,29,"Se conservan; son extremos estadísticos, no er..."
2,heart_ejection_fraction,30.0,45.0,15.0,7.50,67.50,2,"Se conservan; son extremos estadísticos, no er..."
3,platelets_concentration_in_blood,212500.0,303500.0,91000.0,76000.00,440000.00,21,"Se conservan; son extremos estadísticos, no er..."
4,serum_creatinine_concentration_in_blood,0.9,1.4,0.5,0.15,2.15,29,"Se conservan; son extremos estadísticos, no er..."
5,serum_sodium_concentration_in_blood,134.0,140.0,6.0,125.00,149.00,4,"Se conservan; son extremos estadísticos, no er..."
6,days_in_study,73.0,203.0,130.0,-122.00,398.00,0,"Se conservan; son extremos estadísticos, no er..."


### Decisión frente a calidad

- Los controles del conjunto no justifican eliminar registros ni imputar valores cuando el resultado es cero.
- Los valores extremos detectados por IQR se **conservan**, porque el dataset es clínico y la sola rareza estadística no prueba un error de captura.
- Los indicadores booleanos/binarios se normalizan a `0/1` para facilitar cálculos reproducibles.
- No se alteran unidades ni mediciones clínicas.


## 5. Enriquecimiento

In [5]:
# 1. Creamos variables derivadas sobre el DataFrame ya transformado.
# 2. Sirve para facilitar comparaciones y preparar filtros legibles para la futura AE4 en Streamlit.
df_final, variables_enriquecidas = enriquecer(df_transformado)

# 3. Mostramos fórmula, significado y utilidad de cada variable creada.
# 4. Sirve para documentar el enriquecimiento exigido por la rúbrica.
display(variables_enriquecidas)

# 5. Comparamos las dimensiones original y final.
# 6. Sirve para evidenciar que no se perdieron filas y que las columnas adicionales provienen del enriquecimiento.
print("Original:", df_original.shape)
print("Transformado:", df_transformado.shape)
print("Final enriquecido:", df_final.shape)


,variable,regla,significado,utilidad,afectados
0,estado_seguimiento,is_dead: 1=Registró muerte; 0=No registró muerte,Etiqueta legible del desenlace registrado dura...,"Facilita tablas, filtros y futura visualizació...",299
1,grupo_edad,<50; 50-59; 60-69; 70-79; 80+ años,Agrupación descriptiva de edad.,Permite comparar la distribución del evento po...,299
2,condiciones_registradas,anemia + diabetes + presión arterial alta (cad...,Número de las tres condiciones binarias regist...,Resume simultáneamente las condiciones disponi...,299
3,nivel_condiciones_registradas,0; 1; 2 o 3 condiciones registradas,Categoría descriptiva derivada del conteo ante...,Facilita la comparación de proporciones en tab...,299
4,sexo_registrado,is_male: 1=Masculino; 0=Femenino,Etiqueta legible del sexo codificado en la fue...,Evita mostrar códigos 0/1 en resultados y filt...,299
5,tabaquismo_registrado,is_smoker: 1=Sí; 0=No,Etiqueta legible del indicador de tabaquismo r...,Facilita lectura de tablas y futura visualizac...,299


Original: (299, 13)
Transformado: (299, 13)
Final enriquecido: (299, 19)


## 6. Diccionario actualizado

In [6]:
# 1. Construimos automáticamente el diccionario del DataFrame final.
# 2. Sirve para mantener coherencia entre los datos, el documento y las variables derivadas.
diccionario = construir_diccionario(df_final)
display(diccionario)


,variable,significado,tipo_pandas,unidad_o_codificacion
0,age,Edad registrada,int64,años
1,has_anaemia,Indicador de anemia registrada,int8,0/1
2,creatinine_phosphokinase_concentration_in_blood,Concentración de creatina fosfoquinasa en sangre,float64,mcg/L según documentación fuente
3,has_diabetes,Indicador de diabetes registrada,int8,0/1
4,heart_ejection_fraction,Fracción de eyección cardíaca,float64,%
5,has_high_blood_pressure,Indicador de presión arterial alta registrada,int8,0/1
6,platelets_concentration_in_blood,Concentración de plaquetas en sangre,float64,kiloplaquetas/mL según documentación fuente
7,serum_creatinine_concentration_in_blood,Creatinina sérica,float64,mg/dL
8,serum_sodium_concentration_in_blood,Sodio sérico,float64,mEq/L
9,is_male,Sexo masculino codificado en la fuente,int8,0/1


## 7. Respuestas descriptivas y tablas de resultados

In [7]:
# 1. Construimos una tabla específica para cada pregunta operacionalizada.
# 2. Sirve para responder la necesidad con resultados verificables y reutilizables.
tablas = construir_tablas(df_final)

# 3. Mostramos cada tabla con su nombre.
# 4. Sirve para revisar en el notebook exactamente lo que luego se exportará a CSV.
for nombre, tabla in tablas.items():
    print("\n", nombre.upper())
    display(tabla)



 TABLA_01_ESTADO_SEGUIMIENTO


,estado_seguimiento,registros,porcentaje
0,No registró muerte,203,67.89
1,Registró muerte,96,32.11



 TABLA_02_CUANTITATIVAS_POR_ESTADO


,variable,estado_seguimiento,n,media,mediana,desviacion_estandar,coeficiente_variacion_pct,minimo,q1,q3,maximo
0,age,No registró muerte,203,58.759,60.0,10.637,18.10,40.0,50.000,65.00,90.0
1,age,Registró muerte,96,65.208,65.0,13.217,20.27,42.0,55.000,75.00,95.0
2,creatinine_phosphokinase_concentration_in_blood,No registró muerte,203,540.054,245.0,753.800,139.58,30.0,109.000,582.00,5209.0
3,creatinine_phosphokinase_concentration_in_blood,Registró muerte,96,670.198,259.0,1316.581,196.45,23.0,128.750,582.00,7861.0
4,heart_ejection_fraction,No registró muerte,203,40.266,38.0,10.860,26.97,17.0,35.000,45.00,80.0
5,heart_ejection_fraction,Registró muerte,96,33.469,30.0,12.525,37.42,14.0,25.000,38.00,70.0
6,platelets_concentration_in_blood,No registró muerte,203,266657.490,263000.0,97531.202,36.58,25100.0,219500.000,302000.00,850000.0
7,platelets_concentration_in_blood,Registró muerte,96,256381.045,258500.0,98525.683,38.43,47000.0,197500.000,311000.00,621000.0
8,serum_creatinine_concentration_in_blood,No registró muerte,203,1.185,1.0,0.654,55.20,0.5,0.900,1.20,6.1
9,serum_creatinine_concentration_in_blood,Registró muerte,96,1.836,1.3,1.469,79.99,0.6,1.075,1.90,9.4



 TABLA_03_BINARIAS_POR_ESTADO


,variable,indicador,estado_seguimiento,n_grupo,registros_con_indicador_1,porcentaje_en_grupo
0,has_anaemia,Anemia registrada,No registró muerte,203,83,40.89
1,has_anaemia,Anemia registrada,Registró muerte,96,46,47.92
2,has_diabetes,Diabetes registrada,No registró muerte,203,85,41.87
3,has_diabetes,Diabetes registrada,Registró muerte,96,40,41.67
4,has_high_blood_pressure,Presión arterial alta registrada,No registró muerte,203,66,32.51
5,has_high_blood_pressure,Presión arterial alta registrada,Registró muerte,96,39,40.62
6,is_male,Sexo masculino registrado,No registró muerte,203,132,65.02
7,is_male,Sexo masculino registrado,Registró muerte,96,62,64.58
8,is_smoker,Tabaquismo registrado,No registró muerte,203,66,32.51
9,is_smoker,Tabaquismo registrado,Registró muerte,96,30,31.25



 TABLA_04_GRUPO_EDAD_POR_ESTADO


,grupo_edad,total_registros,muertes_registradas,sin_muerte_registrada,porcentaje_con_muerte_registrada
0,<50,47,11,36,23.4
1,50-59,82,20,62,24.39
2,60-69,93,27,66,29.03
3,70-79,52,20,32,38.46
4,80+,25,18,7,72.0



 TABLA_05_CONDICIONES_REGISTRADAS_POR_ESTADO


,nivel_condiciones_registradas,total_registros,muertes_registradas,sin_muerte_registrada,porcentaje_con_muerte_registrada
0,0 condiciones,67,19,48,28.36
1,1 condición,122,35,87,28.69
2,2 o 3 condiciones,110,42,68,38.18


### Criterio de interpretación

Las tablas permiten describir diferencias **observadas en este conjunto**. Una mayor o menor media, frecuencia o proporción en un grupo no demuestra causalidad, no constituye una estimación poblacional y no debe utilizarse como diagnóstico individual. Además, el tiempo de seguimiento varía entre registros, por lo que su comparación requiere especial prudencia.


## 8. Exportación automática de dataset, tablas y resumen TXT

In [8]:
# 1. Exportamos el DataFrame final y todas las evidencias mediante rutas relativas.
# 2. Sirve para regenerar los CSV y el TXT en cada ejecución, tal como exige AE3.
rutas = exportar_evidencias(
    df_final=df_final,
    tablas=tablas,
    control=control_calidad,
    comparacion=comparacion,
    tipos=comparacion_tipos,
    extremos=extremos_iqr,
    enriquecimiento=variables_enriquecidas,
    metadatos=metadatos,
    raiz=RAIZ,
)

# 3. Exportamos también la matriz pregunta-variable-resultado.
# 4. Sirve para dejar trazabilidad de la lógica analítica fuera del notebook.
ruta_preguntas = RAIZ / "AE3" / "resultados" / "preguntas_variables_resultados.csv"
preguntas.to_csv(ruta_preguntas, index=False, encoding="utf-8")

for nombre, ruta in rutas.items():
    print(nombre, "->", ruta.relative_to(RAIZ))
print("preguntas ->", ruta_preguntas.relative_to(RAIZ))


dataset_preparado -> AE3\datos\heart_failure_preparado.csv
tabla_01_estado_seguimiento -> AE3\resultados\tabla_01_estado_seguimiento.csv
tabla_02_cuantitativas_por_estado -> AE3\resultados\tabla_02_cuantitativas_por_estado.csv
tabla_03_binarias_por_estado -> AE3\resultados\tabla_03_binarias_por_estado.csv
tabla_04_grupo_edad_por_estado -> AE3\resultados\tabla_04_grupo_edad_por_estado.csv
tabla_05_condiciones_registradas_por_estado -> AE3\resultados\tabla_05_condiciones_registradas_por_estado.csv
control_calidad -> AE3\resultados\control_calidad.csv
comparacion_antes_despues -> AE3\resultados\comparacion_antes_despues.csv
comparacion_tipos -> AE3\resultados\comparacion_tipos.csv
valores_extremos_iqr -> AE3\resultados\valores_extremos_iqr.csv
variables_enriquecidas -> AE3\resultados\variables_enriquecidas.csv
diccionario_datos_ae3 -> AE3\resultados\diccionario_datos_ae3.csv
resumen_txt -> AE3\resultados\resumen_dataframe_final.txt
preguntas -> AE3\resultados\preguntas_variables_resultado

## 9. Verificaciones finales de coherencia

In [9]:
# 1. Verificamos que se conservaron los 299 registros.
# 2. Sirve para demostrar que ninguna transformación eliminó pacientes.
assert len(df_original) == len(df_final) == 299

# 3. Verificamos que el DataFrame final no tenga valores nulos.
# 4. Sirve para comprobar que la transformación no creó faltantes.
assert int(df_final.isna().sum().sum()) == 0

# 5. Verificamos que los porcentajes de P1 sumen aproximadamente 100 %.
# 6. Sirve para comprobar consistencia básica de la tabla de distribución.
assert abs(tablas["tabla_01_estado_seguimiento"]["porcentaje"].sum() - 100) < 0.02

# 7. Verificamos que cada CSV y el TXT obligatorio existan.
# 8. Sirve para asegurar que la entrega pueda reproducirse antes de publicarla.
for ruta in rutas.values():
    assert ruta.exists() and ruta.stat().st_size > 0, f"Falta evidencia: {ruta}"
assert ruta_preguntas.exists() and ruta_preguntas.stat().st_size > 0

# 9. Mostramos una confirmación final.
# 10. Sirve para dejar una evidencia visible de ejecución completa sin errores.
print("[OK] AE3 ejecutada completamente: transformación, enriquecimiento, tablas y exportaciones validadas.")
print("Dimensión final:", df_final.shape)


[OK] AE3 ejecutada completamente: transformación, enriquecimiento, tablas y exportaciones validadas.
Dimensión final: (299, 19)
